# Lab 06: Writing Data in Neo4j

**Team**: Paola Covarrubias, Darío García, Guillermo Romero

In [1]:
from SparkUtils import SparkUtils

from pyspark.sql.functions import col, lit, initcap
from graphframes import GraphFrame

In [2]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Lab 06: Writing Data in Neo4j"
SPARK_PACKAGES = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"

spark = SparkUtils(MASTER_URL, APP_NAME, spark_packages=SPARK_PACKAGES)._spark

spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-36b691c1-bdab-4ebb-a3c0-8d47ef02b016;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

## Reading the Dataset

In [3]:
columns = [
    ("user", "string"),
    ("prod", "string"),
    ("rating", "float"),
    ("timestamp", "string")
]

schema = SparkUtils.generate_schema(columns)

df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv("/opt/spark/work-dir/data/videogames") \
    .limit(20000)

# df.show(4)

## Creating GraphFrames

In [4]:
users = df\
    .select(col("user").alias("id")) \
    .withColumn("type", lit("user"))

products = df\
    .select(col("prod").alias("id")) \
    .withColumn("type", lit("product"))

vertices = users.union(products).distinct()

edges = df.select(
    col("user").alias("src"),
    col("rating"),
    col("prod").alias("dst")
)

gf = GraphFrame(vertices, edges)

# gf.vertices.show(3)
# gf.edges.show(3)

/opt/spark/python/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


## Graph Analysis

### PageRank

In [5]:
result = gf.pageRank(resetProbability=0.15, maxIter=5)

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [6]:
# ranked vertices
result.vertices\
  .filter(col("type") == "product")\
  .select("id", "type", "pagerank")\
  .orderBy(col("pagerank").desc())\
  .show(10, truncate=False)

+----------+-------+------------------+
|id        |type   |pagerank          |
+----------+-------+------------------+
|B00000JRSB|product|424.3439462285389 |
|B00000K4MC|product|200.61532758942766|
|B00000K2R4|product|176.38480145625368|
|B00000DMB3|product|165.69767554159895|
|B00001IVRD|product|150.6789648043016 |
|B00000J97G|product|110.23105736236893|
|B00000IYEQ|product|105.22639440721103|
|B00000F1GM|product|94.31892838250737 |
|B00001X50M|product|84.15315966115858 |
|B00000J2W7|product|81.50382044078884 |
+----------+-------+------------------+
only showing top 10 rows


### Label Propagation

In [7]:
lp = gf.labelPropagation(maxIter=5)

In [8]:
lp.show(5)

+--------------+----+-----------+
|            id|type|      label|
+--------------+----+-----------+
| AE7GUHCDQQ4UI|user|          0|
|A26B0P6K95SIKW|user|          1|
|A182S3ANC0W7DL|user|34359738369|
|A1T98OCCYW6OBI|user|34359738370|
|A1TBUSGCBTXWFC|user|34359738370|
+--------------+----+-----------+
only showing top 5 rows


### Triangle Counting

In [9]:
tc = gf.triangleCount()

In [10]:
tc.show(5)

+-----+--------------+----+
|count|            id|type|
+-----+--------------+----+
|    0| AE7GUHCDQQ4UI|user|
|    0|A1TBUSGCBTXWFC|user|
|    0|A26B0P6K95SIKW|user|
|    0|A366EUKI8WMGYB|user|
|    0|A182S3ANC0W7DL|user|
+-----+--------------+----+
only showing top 5 rows


### Degree Distribution

In [11]:
# InDegree
in_deg = gf.inDegrees.join(vertices, "id")

In [12]:
in_deg.show(10)

+----------+--------+-------+
|        id|inDegree|   type|
+----------+--------+-------+
|0439339960|       1|product|
|0970154097|       3|product|
|0984529527|       1|product|
|1558843477|       1|product|
|1563820412|       1|product|
|1888449543|       4|product|
|1894353226|       3|product|
|3815864844|       9|product|
|7115127026|       4|product|
|7293000928|       8|product|
+----------+--------+-------+
only showing top 10 rows


In [13]:
out_deg = gf.outDegrees.join(vertices, "id")

In [14]:
out_deg.show(10)

+--------------+---------+----+
|            id|outDegree|type|
+--------------+---------+----+
| AE7GUHCDQQ4UI|        1|user|
|A26B0P6K95SIKW|        1|user|
|A182S3ANC0W7DL|        1|user|
|A1T98OCCYW6OBI|        1|user|
|A1TBUSGCBTXWFC|        1|user|
|A366EUKI8WMGYB|        1|user|
|A129SW886TYQ6H|        1|user|
|A2M64UKVOU9CWU|        1|user|
|A10N7L0GMRODUO|        1|user|
| AFWPLXT2OD6H1|        1|user|
+--------------+---------+----+
only showing top 10 rows


## Writing Data in Neo4j

In [15]:
neo4j_url = "bolt://neo4j-pdm:7687"
neo4j_user = "neo4j"
neo4j_pass = "neo4j-pdm"

In [16]:
# caching
vertices_user = gf.vertices\
  .withColumn("type", initcap(col("type"))) \
  .filter(col("type") == "User") \
  .cache()

vertices_user.count()  # materialize cache

16320

In [17]:
vertices_user.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_pass) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .option("batch.size", 5000) \
  .save()

In [18]:
# caching
vertices_product = gf.vertices\
  .withColumn("type", initcap(col("type"))) \
  .filter(col("type") == "Product") \
  .cache()

vertices_product.count()  # materialize cache

1143

In [19]:

vertices_product.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_pass) \
  .option("labels", ":Product") \
  .option("node.keys", "id") \
  .option("batch.size", 5000) \
  .save()

In [20]:

print(f"{gf.vertices.count()} vertices wrote in Neo4j")

17463 vertices wrote in Neo4j


In [21]:
gf.edges.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_pass) \
  .option("relationship", "RATES") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":Product") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .save()

In [24]:
print(f"{gf.edges.count()} edges wrote in Neo4j")

20000 edges wrote in Neo4j


## Querying the Graph

![Graph visualization](../img/Lab%2006%20-%20graph%20visualization.png)

In [ ]:
# spark.stop()